In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

project_root = Path.cwd().parents[3] # data_eda -> eurostat -> indicators_data -> app -> project root
sys.path.append(str(project_root)) # Lets Python find the 'app' package from inside this notebook
import app.utils.settings as s

# EUROSTAT - Data cleaning

Esta notebook muestra el proceso de limpieza  para cada uno de los indicadores seleccionados de EUROSTAT a partir de las observaciones vistas en el proceso de inspección de datos. Es importante tener en cuenta que solo se van a limpiar datos, cualquier otro proceso relacionado puramente con EDA (p.ej. detectar y gestionar outliers) no forman parte del contexto de esta notebook.

Se tomarán los datasets "raw" de la carpeta 'data_raw/' y, una vez aplicada la limpieza inicial, se guardará una primera versión transformada de cada dataset en la carpeta 'data_interim/' de donde se leerán los datasets de cada indicador limpios con el fin de transformar, combinar y obtener la versión final de los datasets necesarios para pasos posteriores de este proyecto, los cuales se guardarán en 'data_clean/'.

## **nrg_ind_ren**: Cuota de energía renovable sobre el consumo bruto final por país

In [2]:
ind_ren_df = pd.read_csv(r"../data_raw/nrg_ind_ren.csv", sep=',', encoding='utf-8')
ind_ren_df.head(5)

,freq,nrg_bal,unit,geo\TIME_PERIOD,2004,2005,2006,2007,2008,2009,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,A,REN,PC,AL,29.620,31.367,32.070,32.657,32.448,31.437,...,36.953,35.776,36.572,38.042,45.015,41.389,44.076,46.620,43.111,NaN
1,A,REN,PC,AT,22.553,24.353,26.276,28.144,28.788,31.039,...,33.370,33.136,33.784,33.755,36.545,34.643,34.057,41.600,42.951,44.2
2,A,REN,PC,BA,NaN,NaN,NaN,NaN,NaN,NaN,...,25.358,23.241,35.972,37.454,39.835,36.587,39.320,40.038,38.524,NaN
3,A,REN,PC,BE,1.916,2.325,2.658,3.140,3.611,4.746,...,8.744,9.136,9.472,9.929,13.000,13.017,13.831,14.673,14.336,14.9
4,A,REN,PC,BG,9.231,9.173,9.415,9.098,10.345,12.005,...,18.760,18.695,20.581,21.545,23.319,19.394,19.079,23.151,23.210,22.7


El primer paso será filtrar el dataset de forma que únicamente queden los valores relativos al consumo porcentual de renovables sobre el consumo total bruto por país, es decir, aquellos registros para los que 'nrg_bal' sea 'REN'.

In [3]:
ind_ren_df_filtered = (
    ind_ren_df[ind_ren_df['nrg_bal'] == 'REN']
    .copy()
)
ind_ren_df_filtered.shape

(38, 26)

El dataset se ve reducido a una línia por país que presenta registros globales de consumo de renovables.

Se revisan los valores nulos para ver si hay países que se puedan obviar dada la finalidad comparativa de este dataset.

In [4]:
nuls_by_country = ind_ren_df_filtered.drop(columns='geo\\TIME_PERIOD').isna().groupby(ind_ren_df_filtered['geo\\TIME_PERIOD']).sum()
nuls_by_country

,freq,nrg_bal,unit,2004,2005,2006,2007,2008,2009,2010,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
geo\TIME_PERIOD,,,,,,,,,,,,,,,,,,,,,
AL,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
AT,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
BA,0,0,0,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,1
BE,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
BG,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
CY,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
CZ,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
DE,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
DK,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
nuls_by_country.sum(axis=1)

geo\TIME_PERIOD
AL            1
AT            0
BA           11
BE            0
BG            0
CY            0
CZ            0
DE            0
DK            0
EE            0
EL            0
ES            0
EU27_2020     0
FI            0
FR            0
GE           18
HR            0
HU            0
IE            0
IS            1
IT            0
LT            0
LU            0
LV            0
MD            0
ME            1
MK            1
MT            0
NL            0
NO            0
PL            0
PT            0
RO            0
RS            1
SE            0
SI            0
SK            0
XK            1
dtype: int64

Dada la alta proporción de nulos en algunos países y la existencia de categorías que no se relacionan con un país en específico, se decide dejar fuera aquellos países que no forman parte de la UE como lo son Bosnia y Hezergovina o Georgia, los dos casos con más valores nulos aunque no los únicos.

In [6]:
ind_ren_df_filtered = ind_ren_df_filtered[ind_ren_df_filtered['geo\\TIME_PERIOD'].isin(s.eu_countries)]
ind_ren_df_filtered['geo\\TIME_PERIOD'].unique()

array(['AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI',
       'FR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'MT', 'NL', 'PL',
       'PT', 'RO', 'SE', 'SI', 'SK'], dtype=object)

Dado que no hay nulos a gestionar por incapacidad de imputar valores fiables en aquellos casos en que no exista fracción de renovables sobre el consumo total de un país en un año concreto, el único paso restante en esta fase de limpieza será corregir la tipología de algunos datos.

'freq', 'nrg_bal' y 'geo\\TIME_PERIOD' deben ser categoricas, si bien ni la frecuencia ni el tipo de valor mostrado presentan más de una categoría distinta. Por otro lado, si que existen varios países en el dataset.
Las unidades pueden mantenerse como texto.

In [7]:
ind_ren_df_filtered['freq'].unique(), ind_ren_df_filtered['nrg_bal'].unique()

(array(['A'], dtype=object), array(['REN'], dtype=object))

In [8]:
ind_ren_df_filtered['geo\\TIME_PERIOD'] = ind_ren_df_filtered['geo\\TIME_PERIOD'].astype('category')
ind_ren_df_filtered['unit'] = ind_ren_df_filtered['unit'].astype('string')

In [9]:
ind_ren_df_filtered.dtypes

freq                       object
nrg_bal                    object
unit               string[python]
geo\TIME_PERIOD          category
2004                      float64
2005                      float64
2006                      float64
2007                      float64
2008                      float64
2009                      float64
2010                      float64
2011                      float64
2012                      float64
2013                      float64
2014                      float64
2015                      float64
2016                      float64
2017                      float64
2018                      float64
2019                      float64
2020                      float64
2021                      float64
2022                      float64
2023                      float64
2024                      float64
2025                      float64
dtype: object

In [10]:
ind_ren_df_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 27 entries, 1 to 36
Data columns (total 26 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   freq             27 non-null     object  
 1   nrg_bal          27 non-null     object  
 2   unit             27 non-null     string  
 3   geo\TIME_PERIOD  27 non-null     category
 4   2004             27 non-null     float64 
 5   2005             27 non-null     float64 
 6   2006             27 non-null     float64 
 7   2007             27 non-null     float64 
 8   2008             27 non-null     float64 
 9   2009             27 non-null     float64 
 10  2010             27 non-null     float64 
 11  2011             27 non-null     float64 
 12  2012             27 non-null     float64 
 13  2013             27 non-null     float64 
 14  2014             27 non-null     float64 
 15  2015             27 non-null     float64 
 16  2016             27 non-null     float64 
 17  2017

In [11]:
ind_ren_df_filtered['geo\\TIME_PERIOD'].duplicated().sum()

np.int64(0)

In [ ]:
from pathlib import Path
Path("../data_interim").mkdir(parents=True, exist_ok=True)

In [12]:
ind_ren_df_filtered.to_parquet(r"../data_interim/ind_ren_country.parquet", index=False)

## **nrg_pc_204**: Precios de la electricidad para consumo desde 2007

In [13]:
nrg_pc_df = pd.read_csv(r"../data_raw/nrg_pc_204.csv", sep=',', encoding='utf-8')
nrg_pc_df.head(5)

,freq,siec,nrg_cons,unit,tax,currency,geo\TIME_PERIOD,2007-S1,2007-S2,2008-S1,...,2021-S1,2021-S2,2022-S1,2022-S2,2023-S1,2023-S2,2024-S1,2024-S2,2025-S1,2025-S2
0,S,E7000,KWH1000-2499,KWH,I_TAX,EUR,AL,NaN,NaN,NaN,...,0.0925,0.0937,0.0941,0.0976,0.1010,0.1089,0.1114,0.1150,0.1156,0.1175
1,S,E7000,KWH1000-2499,KWH,I_TAX,EUR,AT,NaN,0.2008,0.2005,...,0.2650,0.2787,0.2616,0.2614,0.2988,0.3055,0.3011,0.2631,0.3165,0.3669
2,S,E7000,KWH1000-2499,KWH,I_TAX,EUR,BA,NaN,NaN,NaN,...,0.0980,0.0970,0.0996,0.1003,0.1043,0.0990,0.1067,0.1054,0.1100,0.1125
3,S,E7000,KWH1000-2499,KWH,I_TAX,EUR,BE,NaN,0.1867,0.2172,...,0.2910,0.3161,0.3633,0.4586,0.4559,0.4111,0.3715,0.3695,0.3939,0.3806
4,S,E7000,KWH1000-2499,KWH,I_TAX,EUR,BG,NaN,0.0721,0.0731,...,0.1045,0.1104,0.1110,0.1147,0.1146,0.1192,0.1193,0.1224,0.1313,0.1355


Antes que nada, se aplica el mismo filtro para quitar cualquier país fuera de la UE del estudio.

In [14]:
nrg_pc_df_filtered = (
    nrg_pc_df[
        nrg_pc_df['geo\\TIME_PERIOD'].isin(s.eu_countries)
    ].copy()
)
nrg_pc_df_filtered['geo\\TIME_PERIOD'].unique()

array(['AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI',
       'FR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'MT', 'NL', 'PL',
       'PT', 'RO', 'SE', 'SI', 'SK'], dtype=object)

Si bien el precio siempre viene marcado por KWH, hay tres tipos de categorías dentro de 'currency'; 'EUR' para precio en euros, 'NAC' para el precio en moneda nacional y 'PPS' que indica el 'Purchasing Power Standard' para el poder adquisitivo.

Se elimina NAC porque representa precios en moneda nacional y no permite una comparación directa homogénea entre países. Se mantienen EUR, para comparar precios nominales, y PPS, para comparaciones ajustadas por poder adquisitivo. Además, para Portugal, EUR equivale a NAC.

In [ ]:
nrg_pc_df_filtered = nrg_pc_df_filtered[nrg_pc_df_filtered['currency'].isin(['EUR', 'PPS'])]
nrg_pc_df_filtered['currency'].unique()

C:\Users\roger\AppData\Local\Temp\ipykernel_26444\3384443355.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  nrg_pc_df_filtered = nrg_pc_df_filtered[nrg_pc_df['currency'].isin(['EUR', 'PPS'])]


array(['EUR', 'PPS'], dtype=object)

También se filtra por tipo de impuesto incluido, en este caso nos interesa únicamente el precio con todos los impuestos incluídos 'I_TAX'

In [16]:
nrg_pc_df_filtered = nrg_pc_df_filtered[nrg_pc_df_filtered['tax'] == 'I_TAX']
nrg_pc_df_filtered['tax'].unique()

array(['I_TAX'], dtype=object)

Aplicados los filtros anteriores, cabe únicamente transformar los tipos de datos para las columnas especificadas en el notebook de inspección del dataset.

'freq' y 'siec' se mantienen constantes por lo que no urge transformarlas.

'nrg_cons' será categórica con orden, 'tax', 'currency' y 'geo\TIME_PERIOD' lo serán sin ordenar.

In [17]:
consumption_order = [
    "KWH_LT1000",
    "KWH1000-2499",
    "KWH2500-4999",
    "KWH5000-14999",
    "KWH_GE15000",
    "TOT_KWH"
]


# Nos interesa mantener TOT_KMW para futuros análisis, si bien queda claro que no significa una banda superior a 15000 sino el valor de todas agrupadas

nrg_pc_df_filtered['nrg_cons'] = pd.Categorical(nrg_pc_df_filtered['nrg_cons'], categories=consumption_order, ordered=True)
nrg_pc_df_filtered['nrg_cons'].dtype

CategoricalDtype(categories=['KWH_LT1000', 'KWH1000-2499', 'KWH2500-4999',
                  'KWH5000-14999', 'KWH_GE15000', 'TOT_KWH'],
, ordered=True, categories_dtype=object)

In [18]:
nrg_pc_df_filtered['tax'] = nrg_pc_df_filtered['tax'].astype('category')
nrg_pc_df_filtered['currency'] = nrg_pc_df_filtered['currency'].astype('category')
nrg_pc_df_filtered['geo\\TIME_PERIOD'] = nrg_pc_df_filtered['geo\\TIME_PERIOD'].astype('category')

nrg_pc_df_filtered.dtypes

freq                 object
siec                 object
nrg_cons           category
unit                 object
tax                category
currency           category
geo\TIME_PERIOD    category
2007-S1             float64
2007-S2             float64
2008-S1             float64
2008-S2             float64
2009-S1             float64
2009-S2             float64
2010-S1             float64
2010-S2             float64
2011-S1             float64
2011-S2             float64
2012-S1             float64
2012-S2             float64
2013-S1             float64
2013-S2             float64
2014-S1             float64
2014-S2             float64
2015-S1             float64
2015-S2             float64
2016-S1             float64
2016-S2             float64
2017-S1             float64
2017-S2             float64
2018-S1             float64
2018-S2             float64
2019-S1             float64
2019-S2             float64
2020-S1             float64
2020-S2             float64
2021-S1             

Al no haber modo de imputar valores nulos, de momento se mantienen como NaN a la espera de decidir como gestionarlos de cara a los análisis univariados.

In [19]:
nrg_pc_df_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 324 entries, 1 to 1960
Data columns (total 45 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   freq             324 non-null    object  
 1   siec             324 non-null    object  
 2   nrg_cons         324 non-null    category
 3   unit             324 non-null    object  
 4   tax              324 non-null    category
 5   currency         324 non-null    category
 6   geo\TIME_PERIOD  324 non-null    category
 7   2007-S1          58 non-null     float64 
 8   2007-S2          254 non-null    float64 
 9   2008-S1          266 non-null    float64 
 10  2008-S2          266 non-null    float64 
 11  2009-S1          268 non-null    float64 
 12  2009-S2          268 non-null    float64 
 13  2010-S1          268 non-null    float64 
 14  2010-S2          268 non-null    float64 
 15  2011-S1          268 non-null    float64 
 16  2011-S2          268 non-null    float64 
 17  2

In [20]:
nrg_pc_df_filtered[
    ['geo\\TIME_PERIOD', 'nrg_cons', 'currency']
].duplicated().sum()

np.int64(0)

In [21]:
nrg_pc_df_filtered.to_parquet(r"../data_interim/energy_prices.parquet", index=False)